# Convert MARC to BIBFRAME
Turn a MARC record into Blue Core BIBFRAME with the API's converter (the Library of Congress marc2bibframe2 stylesheet, with Blue Core's policy applied), look at what it produced, and optionally load the result into your local stack.

Converting doesn't store anything, but it's a POST that needs a login, so it runs on your local stack (dev and stage are read-only here). One record per request: bulk MARC goes through the `marc_to_bibframe` workflow instead.

In [ ]:
# Your local stack, or dev / stage: pick it on the Notebooks page.
from bluecore import DASHBOARD_API, show, target
bc = target()
bc

In [ ]:
bc.require_writable("Converting MARC")   # a POST with a login: local stack only
AUTH = bc.auth_headers()

## A MARC record
The API's own MARCXML example. To use yours, set `MARCXML` to its text, or `MARC_FILE` to a `.xml` or `.mrc` file in this notebook's folder.

In [ ]:
import pathlib

MARC_FILE = ""   # e.g. "record.mrc" or "record.xml"
MARCXML = bc.example("/marc2bibframe", "application/xml")

if MARC_FILE:
    body = pathlib.Path(MARC_FILE).read_bytes()
    content_type = "application/xml" if body.lstrip().startswith(b"<") else "application/marc"
else:
    body, content_type = MARCXML.encode(), "application/xml"
print(content_type, len(body), "bytes")

## Binary MARC to MARCXML
`/marc2xml` only takes binary MARC21. `marc21()` builds a small record to try it with.

In [ ]:
from bluecore import marc21

binary = body if content_type == "application/marc" else marc21([
    ("001", "92005291"),
    ("100", "1 ", [("a", "Austen, Jane,"), ("d", "1775-1817.")]),
    ("245", "10", [("a", "Pride and prejudice /"), ("c", "Jane Austen.")]),
])
show(bc.post("/api/marc2xml", data=binary, headers={**AUTH, "Content-Type": "application/marc"}), 800)

## MARC to BIBFRAME
Pick the output with `Accept`: `application/ld+json` (default), `text/turtle`, `application/rdf+xml` or `application/n-triples`. `source_base_uri` sets the base for the URIs it mints (the default is a deliberate placeholder).

In [ ]:
r = bc.post("/api/marc2bibframe", data=body, params={"source_base_uri": "https://example.org/marc/"},
            headers={**AUTH, "Content-Type": content_type, "Accept": "text/turtle"})
print(r.status_code)
print(r.text[:3000])

## What it produced
The resources in the result, by type.

In [ ]:
import collections
import pandas as pd

r = bc.post("/api/marc2bibframe", data=body, headers={**AUTH, "Content-Type": content_type})
graph = r.json()
types = collections.Counter(t.rsplit("/", 1)[-1] for node in graph for t in node.get("@type", []))
print(len(graph), "nodes")
pd.Series(types, name="nodes").sort_values(ascending=False)

## Load it into your stack (optional)
Set `LOAD = True` to load the result through a batch (the `resource_loader` DAG), wait for it, and list the records it made. The **Load records with a batch** starter explains batches.

In [ ]:
import json, time

LOAD = False
if LOAD:
    r = bc.post("/api/batches/upload/", files={"file": ("converted.jsonld", json.dumps(graph), "application/ld+json")}, headers=AUTH)
    show(r)
    state, tasks = bc.wait_for_run("resource_loader", r.json()["workflow_id"])
    print(state, tasks)
    print(bc.get("/api/change_documents/works/feed").json()["totalItems"], "work changes in the feed now")
else:
    print("Set LOAD = True to load it.")